# 02 – Baseline Models (Traditional ML)

**Project:** Multi-Agent DRL + CNN Alternative Data for Credit Decisioning

This notebook trains Logistic Regression, Random Forest and XGBoost on the German Credit tabular features (with thin-file flag) and evaluates performance, especially on the thin-file segment (RQ4).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed – will skip XGB model")

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
%matplotlib inline
print("Libraries loaded.")

## 1. Load Processed Data

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "german_credit_model.csv")
print(f"Shape: {df.shape}")
print(f"Default rate: {df['default'].mean():.2%}")
print(f"Thin-file rate: {df['thin_file_flag'].mean():.2%}")

X = df.drop(columns=['default'])
y = df['default']
thin = df['thin_file_flag']

X_train, X_test, y_train, y_test, thin_train, thin_test = train_test_split(
    X, y, thin, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Train Baseline Models

In [ ]:
models = {}
results = []

# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_s, y_train)
models['Logistic Regression'] = lr

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = rf

# XGBoost
if HAS_XGB:
    xgb = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                        random_state=42, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    models['XGBoost'] = xgb

print("Models trained:", list(models.keys()))

## 3. Evaluation

In [ ]:
def evaluate(name, model, X_te, y_te, thin_te, scaled=False):
    X_in = X_te if not scaled else X_test_s
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_in)[:, 1]
    else:
        proba = model.decision_function(X_in)
        proba = (proba - proba.min()) / (proba.max() - proba.min() + 1e-8)
    pred = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_te, proba)
    f1  = f1_score(y_te, pred)
    rec = recall_score(y_te, pred)
    prec = precision_score(y_te, pred)

    # Thin-file metrics
    mask = thin_te == 1
    thin_auc = roc_auc_score(y_te[mask], proba[mask]) if mask.sum() > 1 else np.nan
    thin_approval = (pred[mask] == 0).mean()  # predicted Good

    return {
        'Model': name, 'AUC': auc, 'F1': f1, 'Recall': rec, 'Precision': prec,
        'Thin-file AUC': thin_auc, 'Thin-file Approval Rate': thin_approval
    }

rows = []
for name, model in models.items():
    scaled = (name == 'Logistic Regression')
    rows.append(evaluate(name, model, X_test, y_test, thin_test, scaled=scaled))

res_df = pd.DataFrame(rows)
print(res_df.round(3).to_string(index=False))
res_df.to_csv(RESULTS / "baseline_metrics.csv", index=False)
print("\nSaved → results/baseline_metrics.csv")

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    X_in = X_test_s if name == 'Logistic Regression' else X_test
    RocCurveDisplay.from_estimator(model, X_in, y_test, ax=ax, name=name)
ax.plot([0,1], [0,1], 'k--', label='Random')
ax.set_title('ROC Curves – Baseline Models')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "baseline_roc.png", dpi=120, bbox_inches='tight')
plt.show()

## 4. Feature Importance (Random Forest)

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
imp.plot(kind='barh', color='#3498db')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(RESULTS / "feature_importance.png", dpi=120, bbox_inches='tight')
plt.show()

## 5. Summary for Research Questions

- **RQ1 / RQ2 baseline**: These traditional models set the performance floor that single-agent and multi-agent DRL + CNN must beat.
- **RQ4**: Thin-file AUC and approval rate are key inclusion metrics. Current baselines typically under-serve thin-file customers.

Next → `03_CNN_Encoder.ipynb`